# Phase 2 Acceptance Notebook

**Purpose:** Validate the Phase 2 backtest engine, three reference strategies, and verification utilities end-to-end using the synthetic market fixture.

**Switch to real data:** Replace `build_synthetic_market(...)` calls with a real `DataRepository` instance. All other code is unchanged.

> **DISCLAIMER:** Results are historical backtest, not investment advice.

In [ ]:
# Cell 1: Environment / config hash / code version prelude
import importlib
import sys
from datetime import date
from decimal import Decimal

import pandas as pd

import ah_research
from ah_research.backtest import (
    BacktestConfig,
    CostModelBundle,
    run_backtest,
    verify,
)
from ah_research.backtest.types import hash_config
from ah_research.strategies import (
    AHPremiumMeanReversionStrategy,
    DividendYieldStrategy,
    ValueFactorStrategy,
)

# Synthetic market fixture (no network required)
sys.path.insert(0, "tests")  # make tests/fixtures importable
from fixtures.phase2.synthetic_market import build_synthetic_market

# Backtest date range for all cells
BT_START = date(2022, 1, 1)
BT_END = date(2023, 12, 31)
SYMBOLS = [f"{str(i).zfill(6)}.SH" for i in range(600000, 600010)]

repo = build_synthetic_market(start=BT_START, end=BT_END, symbols=SYMBOLS)

BASE_CFG = BacktestConfig(
    start=BT_START,
    end=BT_END,
    initial_cash=Decimal("1_000_000"),
    benchmark="zero",
    cost_model=CostModelBundle(models={}),
)

code_version = ah_research.__version__ if hasattr(ah_research, "__version__") else "dev"
config_hash = hash_config(BASE_CFG)
data_snapshot_date = str(BT_END)

print(f"Python: {sys.version.split()[0]}")
print(f"Code version : {code_version}")
print(f"Config hash  : {config_hash}")
print(f"Data snapshot: {data_snapshot_date}")
print(f"Symbols ({len(SYMBOLS)}): {SYMBOLS[:3]} ...")

## Cell 2: Run each strategy on the synthetic market

ValueFactor, DividendYield, and AHPremiumMeanReversion are each run over the same 2-year window. Metrics are printed below.

In [ ]:
# Cell 2: Run each of the three reference strategies
results = {}

# ValueFactorStrategy
vf_strat = ValueFactorStrategy(quantile=0.3, max_weight=0.15)
results["value_factor"] = run_backtest(vf_strat, repo, BASE_CFG)
print("=== ValueFactorStrategy ===")
print(results["value_factor"].metrics)
print()

# DividendYieldStrategy
dy_strat = DividendYieldStrategy(quantile=0.3, max_weight=0.15)
results["dividend_yield"] = run_backtest(dy_strat, repo, BASE_CFG)
print("=== DividendYieldStrategy ===")
print(results["dividend_yield"].metrics)
print()

# AHPremiumMeanReversionStrategy (needs HK symbols)
AH_SYMBOLS = ["601318.SH", "2318.HK"]
ah_repo = build_synthetic_market(start=BT_START, end=BT_END, symbols=AH_SYMBOLS)
ah_strat = AHPremiumMeanReversionStrategy()
ah_cfg = BacktestConfig(
    start=BT_START,
    end=BT_END,
    initial_cash=Decimal("1_000_000"),
    benchmark="zero",
    cost_model=CostModelBundle(models={}),
    allow_short=True,
)
results["ah_mr"] = run_backtest(ah_strat, ah_repo, ah_cfg)
print("=== AHPremiumMeanReversionStrategy ===")
print(results["ah_mr"].metrics)

## Cell 3: Equity curves (log y-axis) vs benchmark

Benchmark is `zero` (flat at initial NAV = 1,000,000) for the synthetic run.

In [ ]:
# Cell 3: Equity curves — text summary (matplotlib optional)
for name, result in results.items():
    ec = result.equity_curve
    print(
        f"{name}: start={ec.iloc[0]:.0f}, end={ec.iloc[-1]:.0f}, "
        f"days={len(ec)}, total_return={(ec.iloc[-1] / ec.iloc[0] - 1) * 100:.1f}%"
    )

try:
    import matplotlib

    matplotlib.use("Agg")  # non-interactive backend for CI
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, result) in zip(axes, results.items(), strict=False):
        ec = result.equity_curve / result.equity_curve.iloc[0]
        bm = result.benchmark_curve / result.benchmark_curve.iloc[0]
        ax.semilogy(ec.index, ec.values, label="Strategy")
        ax.semilogy(bm.index, bm.values, label="Benchmark", linestyle="--", alpha=0.6)
        ax.set_title(name)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print("[Plot displayed]")
except ImportError:
    print("[matplotlib not installed — skipping equity curve plot]")

## Cell 4: Drawdown analysis

In [ ]:
# Cell 4: Drawdown — text summary and optional plot
from ah_research.backtest.metrics import max_drawdown

for name, result in results.items():
    mdd, duration = max_drawdown(result.equity_curve)
    print(f"{name}: max_drawdown={mdd * 100:.1f}%, duration={duration}d")

try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, result) in zip(axes, results.items(), strict=False):
        ec = result.equity_curve
        peak = ec.cummax()
        dd = (ec - peak) / peak * 100
        ax.fill_between(dd.index, dd.values, 0, alpha=0.4, color="red")
        ax.set_title(f"{name} drawdown (%)")
    plt.tight_layout()
    plt.show()
    print("[Plot displayed]")
except ImportError:
    print("[matplotlib not installed — skipping drawdown plot]")

## Cell 5: `verify.leakage_canary` — three canary types

PASS = no future-data leakage detected. FAIL = leakage suspected.

In [ ]:
# Cell 5: leakage_canary for each strategy
canary_strategies = [
    ("value_factor", vf_strat, repo, BASE_CFG),
    ("dividend_yield", dy_strat, repo, BASE_CFG),
    ("ah_mr", ah_strat, ah_repo, ah_cfg),
]

for name, strat, r, cfg in canary_strategies:
    report = verify.leakage_canary(strat, r, cfg)
    print(f"--- {name} ---")
    for cr in report.results:
        status = "PASS" if cr.passed else ("FAIL" if cr.passed is False else "N/A")
        print(f"  [{status}] {cr.kind}: {cr.message}")
    print(f"  all_pass={report.all_pass}")
    print()

## Cell 6: `verify.survivorship_check` — PIT vs static vs random

In [ ]:
# Cell 6: survivorship_check for value_factor strategy
surv_report = verify.survivorship_check(
    ValueFactorStrategy(quantile=0.3, max_weight=0.15),
    repo,
    config=BASE_CFG,
    n_random_universes=5,  # reduced for speed in synthetic fixture
)
print("=== Survivorship Check (ValueFactorStrategy) ===")
print(f"PIT Sharpe            : {surv_report.pit_metrics.sharpe:.3f}")
print(f"Static Sharpe         : {surv_report.static_metrics.sharpe:.3f}")
rdf = surv_report.random_metrics_distribution
if not rdf.empty and "sharpe" in rdf.columns:
    import numpy as np

    random_sharpes = rdf["sharpe"].dropna().values
    print(f"Random mean Sharpe    : {np.mean(random_sharpes):.3f}")
print(
    f"PIT Sharpe percentile : {surv_report.pit_sharpe_percentile:.1f}th within random distribution"
)

## Cell 7: `verify.walk_forward` — 5-split expanding table

In [ ]:
# Cell 7: walk_forward for value_factor strategy (5 splits, expanding)
wf_report = verify.walk_forward(
    lambda: ValueFactorStrategy(quantile=0.3, max_weight=0.15),
    repo,
    start=BT_START,
    end=BT_END,
    config_template=BASE_CFG,
    n_splits=5,
    mode="expanding",
)
print("=== Walk-Forward (expanding, 5 splits) ===")
rows = []
for i, split in enumerate(wf_report.splits):
    rows.append(
        {
            "split": i + 1,
            "is_start": split.is_start,
            "is_end": split.is_end,
            "oos_start": split.oos_start,
            "oos_end": split.oos_end,
            "is_sharpe": round(split.is_metrics.sharpe, 3),
            "oos_sharpe": round(split.oos_metrics.sharpe, 3),
            "is_cagr": round(split.is_metrics.cagr, 3),
            "oos_cagr": round(split.oos_metrics.cagr, 3),
        }
    )
wf_df = pd.DataFrame(rows)
print(wf_df.to_string(index=False))
print(f"Combined OOS Sharpe: {wf_report.combined_oos_metrics.sharpe:.3f}")

## Cell 8: `verify.sensitivity` — parameter sweep for ValueFactorStrategy

In [ ]:
# Cell 8: sensitivity over quantile for value_factor strategy
sens_report = verify.sensitivity(
    lambda quantile: ValueFactorStrategy(quantile=quantile, max_weight=0.15),
    repo,
    config_template=BASE_CFG,
    param_grid={"quantile": [0.1, 0.2, 0.3, 0.4, 0.5]},
)
print("=== Sensitivity: ValueFactorStrategy quantile sweep ===")
display_cols = ["quantile", "cagr", "sharpe", "max_drawdown", "annualized_turnover"]
available = [c for c in display_cols if c in sens_report.grid_df.columns]
print(sens_report.grid_df[available].to_string(index=False))

## Cell 9: AH pair case study — premium z-score with entry/exit markers

Shows the rolling z-score of the A/H premium for a synthetic pair, with entry (z < -2) and exit (|z| < 0.5) markers.

In [ ]:
# Cell 9: AH pair case study — premium + z-score plot
from ah_research.model.types import AHPair, parse_symbol

pair = AHPair(
    a_symbol=parse_symbol("601318.SH"),
    h_symbol=parse_symbol("2318.HK"),
    name_en="Synthetic Pair",
    name_zh="Synthetic",
)
premium_df = ah_repo.compute_ah_premium(pair, BT_START, BT_END)

if premium_df.empty:
    print("No premium data available for this pair in the synthetic fixture.")
else:
    # Compute rolling 60-day z-score
    prem = premium_df.set_index("date")["premium"]
    roll_mean = prem.rolling(60, min_periods=10).mean()
    roll_std = prem.rolling(60, min_periods=10).std()
    zscore = (prem - roll_mean) / roll_std.replace(0, float("nan"))

    entry_dates = zscore[zscore < -2.0].index
    exit_dates = zscore[zscore.abs() < 0.5].index

    print(f"Premium series: {len(prem)} trading days")
    print(f"Entry signals (z < -2.0): {len(entry_dates)}")
    print(f"Exit  signals (|z| < 0.5): {len(exit_dates)}")

    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
        ax1.plot(prem.index, prem.values, color="steelblue", label="AH premium")
        ax1.axhline(0, color="black", linewidth=0.5)
        ax1.set_ylabel("Premium")
        ax1.set_title(f"{pair.name_en}: AH premium")
        ax1.legend()
        ax2.plot(zscore.index, zscore.values, color="darkorange", label="z-score")
        ax2.axhline(-2.0, color="green", linestyle="--", label="Entry threshold")
        ax2.axhline(0.5, color="red", linestyle="--", label="Exit threshold")
        ax2.axhline(-0.5, color="red", linestyle="--")
        ax2.scatter(
            entry_dates,
            zscore[entry_dates],
            marker="^",
            color="green",
            zorder=5,
            s=40,
            label="Entry",
        )
        ax2.scatter(
            exit_dates,
            zscore[exit_dates],
            marker="v",
            color="red",
            zorder=5,
            s=20,
            alpha=0.5,
            label="Exit",
        )
        ax2.set_ylabel("Z-score")
        ax2.legend(fontsize=8)
        plt.tight_layout()
        plt.show()
        print("[Plot displayed]")
    except ImportError:
        print("[matplotlib not installed — skipping premium z-score plot]")

## Final: Reproducibility block

> **DISCLAIMER:** Results are historical backtest, not investment advice.

In [ ]:
# Final cell: reproducibility block
import importlib.metadata

print("=== Reproducibility Block ===")
print(f"code_version     : {code_version}")
print(f"config_hash      : {hash_config(BASE_CFG)}")
print(f"data_snapshot    : {data_snapshot_date}")
print(f"bt_start         : {BT_START}")
print(f"bt_end           : {BT_END}")

for pkg in ["pandas", "numpy", "statsmodels", "pandera", "duckdb"]:
    try:
        ver = importlib.metadata.version(pkg)
        print(f"{pkg:20s}: {ver}")
    except Exception:
        pass

print()
print("DISCLAIMER: Results are historical backtest, not investment advice.")